Convert SQL commands into notebook cells for Marc Jacobs Beauty Revival.
*Co-authored with CoCo*

In [ ]:
%%sql -r use_database_result
USE DATABASE MARCJACOBS_BEAUTY_REVIVAL;

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE VIEW ANALYTICS.ASSORTMENT_BY_CATEGORY AS
SELECT
  era,
  category,
  COUNT(*) AS product_count,
  ROUND(AVG(price_usd), 2) AS avg_price_usd,
  SUM(shade_count) AS total_commercial_shade_count
FROM ANALYTICS.PRODUCTS
GROUP BY era, category;

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE VIEW ANALYTICS.SHADE_COLOR_DISTRIBUTION AS
SELECT
  era,
  category,
  color_family,
  COUNT(*) AS shade_rows
FROM ANALYTICS.SHADES
GROUP BY era, category, color_family;

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE VIEW ANALYTICS.FINISH_MIX_BY_ERA AS
SELECT
  era,
  finish_group,
  COUNT(*) AS shade_rows
FROM ANALYTICS.SHADES
GROUP BY era, finish_group;

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE VIEW ANALYTICS.OOS_BY_PRODUCT AS
SELECT
  era,
  product_name,
  COUNT(*) AS shade_rows,
  SUM(CASE WHEN is_oos THEN 1 ELSE 0 END) AS oos_shade_rows,
  ROUND(
    SUM(CASE WHEN is_oos THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0),
    3
  ) AS oos_share
FROM ANALYTICS.SHADES
GROUP BY era, product_name;

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE VIEW ANALYTICS.COMPLEXION_SHADE_DEPTH AS
SELECT
  era,
  product_name,
  product_type,
  shade_depth_group,
  undertone_inferred_cmyk,
  undertone_confidence,
  COUNT(*) AS shade_rows
FROM ANALYTICS.SHADES
WHERE shade_function IN ('skin_tone', 'bronzer', 'corrector', 'brightener')
GROUP BY
  era,
  product_name,
  product_type,
  shade_depth_group,
  undertone_inferred_cmyk,
  undertone_confidence;

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE VIEW ANALYTICS.COMPLEXION_UNDERTONE_MATRIX AS
SELECT
  era,
  shade_depth_group,
  SUM(CASE WHEN undertone_inferred_cmyk = 'golden_warm' THEN shade_rows ELSE 0 END) AS golden_warm_rows,
  SUM(CASE WHEN undertone_inferred_cmyk = 'neutral_warm' THEN shade_rows ELSE 0 END) AS neutral_warm_rows,
  SUM(shade_rows) AS total_rows
FROM ANALYTICS.COMPLEXION_SHADE_DEPTH
WHERE (era = 'original_launch' AND product_type IN ('Foundation', 'Concealer', 'Bronzer'))
   OR (era = 'revival' AND product_type = 'Bronzer')
GROUP BY era, shade_depth_group
ORDER BY era,
  CASE shade_depth_group
    WHEN 'very_fair' THEN 1
    WHEN 'fair' THEN 2
    WHEN 'light' THEN 3
    WHEN 'light_medium' THEN 4
    WHEN 'medium' THEN 5
    WHEN 'medium_deep' THEN 6
    WHEN 'deep' THEN 7
    WHEN 'very_deep' THEN 8
    WHEN 'universal' THEN 9
    WHEN 'brightener' THEN 10
    WHEN 'corrector' THEN 11
    ELSE 12
  END;